# Trích xuất Tên Quán và Địa Điểm từ Video với Vintern-1B (Bản Fix Lỗi Kết Luận & Chống Nhiễu OCR)
### (Context Caching & Sequential Frame Processing Pipeline - v2)

Mô hình sử dụng: `5CD-AI/Vintern-1B-v3_5` (Multimodal LLM kết hợp InternViT-300M + Qwen2-0.5B-Instruct, tối ưu chuyên biệt cho tiếng Việt và OCR bảng hiệu).

---

### 🔧 Các nâng cấp cốt lõi trong phiên bản v2:
1. **Ưu tiên đọc ảnh gốc không có Bounding Box (`selected_text_frames`)**: Loại bỏ hoàn toàn hiện tượng đọc nhầm các chỉ số confidence/area (`0.81`, `10.75%`, `Area: 17.45%`) trên ảnh debug.
2. **Bộ lọc hậu xử lý OCR (`clean_ocr_val`)**: Loại bỏ triệt để hiện tượng Prompt Echo (`"BẮT BUỘC"`), địa chỉ hallucination mặc định của Qwen2-0.5B (`"123 Nguyễn Văn Cừ"`, `"123 Nguyễn Du"`), câu mô tả chung chung và chuỗi chỉ chứa số.
3. **Chuẩn hóa số tiếng Việt đầy đủ (`normalize_stt_text`)**: Chuyển đổi chính xác các số hàng trăm, hàng chục, năm sinh (`năm trăm tám mươi sáu` -> `586`, `một chín tám bảy` -> `1987`) giúp đồng bộ giữa văn bản STT và chữ trên biển hiệu.
4. **Đảo chiều ưu tiên Prompt & Loại bỏ neo `null` ở `conclude()`**: Đưa dữ liệu OCR đã làm sạch lên đầu (Nguồn 1), chỉ đưa thông tin STT khi thực sự tồn tại (tránh Positional Bias vào chữ `null` khiến LLM 0.5B trả về rỗng).
5. **Cơ chế Majority Voting & Scoring Fallback từ OCR (`_select_best_ocr_candidate`)**: Khi LLM trả về `None` hoặc chọn nhầm câu mô tả dài từ lời thoại, hệ thống tự động bầu chọn và ưu tiên kết quả OCR chuẩn xác nhất từ các khung hình.

In [1]:
# Cài đặt transformers < 5.0.0 (tương thích hoàn toàn kiến trúc InternVL2 / Vintern-1B và Python 3.10 - 3.13)
!pip install -q "transformers<5.0.0,>=4.48" accelerate torchvision torch opencv-python pillow peft timm einops sentencepiece

import os
import re
import gc
import json
import torch
import cv2
import numpy as np
from collections import Counter
from PIL import Image
from typing import List, Dict, Optional, Union
import torchvision.transforms as T
from torchvision.transforms.functional import InterpolationMode
from transformers import AutoModel, AutoTokenizer
from transformers.configuration_utils import PretrainedConfig
from transformers.modeling_utils import PreTrainedModel

# Patch khắc phục các lỗi không tương thích giữa Transformers và kiến trúc InternVL
_orig_to_diff_dict = PretrainedConfig.to_diff_dict
PretrainedConfig.to_diff_dict = lambda self: (
    _orig_to_diff_dict(self) if not hasattr(self, 'llm_config') or self.llm_config else self.to_dict()
)
if not hasattr(PreTrainedModel, "all_tied_weights_keys"):
    PreTrainedModel.all_tied_weights_keys = property(lambda self: getattr(self, "_tied_weights_keys", {}))

print("PyTorch version:", torch.__version__)
print("CUDA khả dụng:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Thiết bị:", torch.cuda.get_device_name(0))

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.41.0 requires huggingface-hub<2.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
PyTorch version: 2.11.0+cu130
CUDA khả dụng: True
Thiết bị: Tesla T4


In [2]:
# --- TIỀN XỬ LÝ ẢNH ĐỘ PHÂN GIẢI ĐỘNG (DYNAMIC RESOLUTION CHO VINTERN-1B) ---
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

def build_transform(input_size=448):
    '''Transform chuẩn hóa ảnh về tensor theo chuẩn ImageNet'''
    return T.Compose([
        T.Lambda(lambda img: img.convert('RGB') if img.mode != 'RGB' else img),
        T.Resize((input_size, input_size), interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
    ])

def find_closest_aspect_ratio(aspect_ratio, target_ratios, width, height, image_size):
    best_ratio_diff = float('inf')
    best_ratio = (1, 1)
    area = width * height
    for ratio in target_ratios:
        target_aspect_ratio = ratio[0] / ratio[1]
        ratio_diff = abs(aspect_ratio - target_aspect_ratio)
        if ratio_diff < best_ratio_diff:
            best_ratio_diff = ratio_diff
            best_ratio = ratio
        elif ratio_diff == best_ratio_diff:
            if area > 0.5 * image_size * image_size * ratio[0] * ratio[1]:
                best_ratio = ratio
    return best_ratio

def dynamic_preprocess(image: Image.Image, min_num=1, max_num=3, image_size=448, use_thumbnail=True):
    '''
    Chia khung hình thành các patch 448x448 để bảo toàn độ nét của chữ biển hiệu.
    Lưu ý: max_num=3 giúp giữ chi tiết cao nhưng tiết kiệm token (~256-512 tokens/ảnh),
    hoàn toàn an toàn trong context window khi xử lý từng frame!
    '''
    orig_width, orig_height = image.size
    aspect_ratio = orig_width / orig_height

    target_ratios = set(
        (i, j) for n in range(min_num, max_num + 1)
        for i in range(1, n + 1) for j in range(1, n + 1)
        if i * j <= max_num and i * j >= min_num
    )
    target_ratios = sorted(target_ratios, key=lambda x: x[0] * x[1])

    target_aspect_ratio = find_closest_aspect_ratio(
        aspect_ratio, target_ratios, orig_width, orig_height, image_size
    )

    target_width = image_size * target_aspect_ratio[0]
    target_height = image_size * target_aspect_ratio[1]
    blocks = target_aspect_ratio[0] * target_aspect_ratio[1]

    resize_img = image.resize((target_width, target_height))
    processed_images = []
    for i in range(blocks):
        box = (
            (i % target_aspect_ratio[0]) * image_size,
            (i // target_aspect_ratio[0]) * image_size,
            ((i % target_aspect_ratio[0]) + 1) * image_size,
            ((i // target_aspect_ratio[0]) + 1) * image_size
        )
        split_img = resize_img.crop(box)
        processed_images.append(split_img)

    if use_thumbnail and len(processed_images) != 1:
        thumbnail_img = image.resize((image_size, image_size))
        processed_images.append(thumbnail_img)

    return processed_images

In [3]:
# --- TIỆN ÍCH TRÍCH XUẤT VÀ TẢI KHUNG HÌNH CHỌN LỌC ---

def load_selected_frames_from_dir(dir_path: str, max_frames: int = 5) -> List[Image.Image]:
    """Tải danh sách ảnh từ thư mục chứa frame đã chọn lọc"""
    if not os.path.exists(dir_path):
        return []
        
    valid_exts = {".jpg", ".jpeg", ".png", ".webp"}
    files = sorted([
        os.path.join(dir_path, f) for f in os.listdir(dir_path)
        if os.path.splitext(f.lower())[1] in valid_exts
    ])
    
    frames = []
    for f in files[:max_frames]:
        try:
            data = np.fromfile(f, dtype=np.uint8)
            img_bgr = cv2.imdecode(data, cv2.IMREAD_COLOR)
            if img_bgr is not None:
                img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
                frames.append(Image.fromarray(img_rgb))
        except Exception as e:
            print(f"Lỗi đọc ảnh {f}: {e}")
    return frames

def extract_keyframes_from_video(video_path: str, max_keyframes: int = 4, sample_fps: float = 1.0) -> List[Image.Image]:
    """Trích xuất nhanh các khung hình có xác suất chứa bảng hiệu chữ cao nhất từ file video .mp4"""
    if not os.path.exists(video_path):
        print(f"Không tìm thấy file video: {video_path}")
        return []
        
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Không thể mở video {video_path}")
        return []
        
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    step = max(1, int(fps / sample_fps))
    candidates = []
    frame_idx = 0
    
    while True:
        ret, frame = cap.read()
        if not ret: break
        if frame_idx % step == 0:
            h, w = frame.shape[:2]
            small = cv2.resize(frame, (w // 2, h // 2))
            gray = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY)
            grad_x = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
            score = float(np.mean(np.abs(grad_x)))
            candidates.append((score, cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))
        frame_idx += 1
    cap.release()
    
    if not candidates:
        return []

    candidates.sort(key=lambda x: x[0], reverse=True)
    return [Image.fromarray(c[1]) for c in candidates[:max_keyframes]]

def get_video_keyframes(video_path: str, max_keyframes: int = 4, video_id: Optional[str] = None, workspace_dir: str = None) -> List[Image.Image]:
    """
    Tiện ích tổng hợp nạp khung hình:
    1. Ưu tiên đọc từ thư mục ảnh sạch `selected_text_frames` (không vẽ đè bounding box/confidence score)
    2. Nếu không có mới fallback sang `selected_text_frames_drawn`
    3. Nếu chưa có, trích xuất trực tiếp từ file video .mp4
    """
    if video_id and workspace_dir:
        # Ưu tiên ảnh sạch không bị vẽ đè số phần trăm / điểm số OCR
        clean_dirs = [
            os.path.join(workspace_dir, "selected_text_output", str(video_id), "selected_text_frames"),
            os.path.join(workspace_dir, "selected_text_output", str(video_id), "frames"),
            os.path.join(workspace_dir, "selected_text_output", str(video_id), "selected_text_frames_drawn")
        ]
        for frame_dir in clean_dirs:
            if os.path.exists(frame_dir):
                cached_frames = load_selected_frames_from_dir(frame_dir, max_frames=max_keyframes)
                if cached_frames:
                    return cached_frames
                
    if os.path.exists(video_path):
        return extract_keyframes_from_video(video_path, max_keyframes=max_keyframes)
        
    return []

In [4]:
# =============================================================================
# TIỆN ÍCH CHUẨN HÓA VĂN BẢN STT, LỌC NHIỄU OCR & PARSE JSON AN TOÀN
# =============================================================================

def normalize_stt_text(text: str) -> str:
    """Chuyển đổi các số viết bằng chữ trong STT tiếng Việt sang chữ số (hỗ trợ hàng trăm, chục, đơn vị)"""
    if not text:
        return ""
    
    digit_map = {
        "không": "0", "linh": "0", "lẻ": "0",
        "một": "1", "mốt": "1", "hai": "2", "ba": "3", "bốn": "4", "tư": "4",
        "năm": "5", "lăm": "5", "sáu": "6", "bảy": "7", "bẩy": "7", "tám": "8", "chín": "9"
    }
    
    normalized = text
    
    # 1. Xử lý cụm 4 chữ số đọc rời (VD: "một chín tám bảy" -> "1987")
    pattern_4digits = r'\b(một|hai)\s+(không|chín|tám)\s+(không|một|hai|ba|bốn|năm|sáu|bảy|tám|chín)\s+(không|một|mốt|hai|ba|bốn|tư|năm|lăm|sáu|bảy|tám|chín)\b'
    def repl_4d(m):
        return "".join(digit_map[w.lower()] for w in m.groups())
    normalized = re.sub(pattern_4digits, repl_4d, normalized, flags=re.IGNORECASE)
    
    # 2. Xử lý số hàng trăm: "<chữ> trăm <chữ> mươi <chữ>" (VD: "năm trăm tám mươi sáu" -> "586")
    pattern_hundreds_full = r'\b(một|hai|ba|bốn|năm|sáu|bảy|tám|chín)\s+trăm\s+(mười|hai|ba|bốn|năm|sáu|bảy|tám|chín)(?:\s+mươi)?(?:\s+(một|mốt|hai|ba|bốn|tư|năm|lăm|sáu|bảy|tám|chín))?\b'
    def repl_hundreds(m):
        h = digit_map[m.group(1).lower()]
        t_word = m.group(2).lower()
        t = "1" if t_word == "mười" else digit_map[t_word]
        u = digit_map[m.group(3).lower()] if m.group(3) else "0"
        return f"{h}{t}{u}"
    normalized = re.sub(pattern_hundreds_full, repl_hundreds, normalized, flags=re.IGNORECASE)

    # 3. Xử lý số hàng trăm có "linh/lẻ": "<chữ> trăm linh/lẻ <chữ>" (VD: "hai trăm linh năm" -> "205")
    pattern_hundreds_zero = r'\b(một|hai|ba|bốn|năm|sáu|bảy|tám|chín)\s+trăm\s+(?:linh|lẻ)\s+(một|hai|ba|bốn|năm|sáu|bảy|tám|chín)\b'
    normalized = re.sub(
        pattern_hundreds_zero,
        lambda m: f"{digit_map[m.group(1).lower()]}0{digit_map[m.group(2).lower()]}",
        normalized,
        flags=re.IGNORECASE
    )

    # 4. Xử lý số hàng chục: "<chữ> mươi <chữ>" (VD: "ba mươi tư" -> "34", "ba mươi" -> "30")
    pattern_tens = r'\b(hai|ba|bốn|năm|sáu|bảy|tám|chín)\s+mươi(?:\s+(mốt|một|hai|ba|bốn|tư|năm|lăm|sáu|bảy|tám|chín))?\b'
    def repl_tens(m):
        t = digit_map[m.group(1).lower()]
        u = digit_map[m.group(2).lower()] if m.group(2) else "0"
        return f"{t}{u}"
    normalized = re.sub(pattern_tens, repl_tens, normalized, flags=re.IGNORECASE)

    # 5. Xử lý "mười <chữ>" (VD: "mười lăm" -> "15")
    pattern_teens = r'\bmười(?:\s+(một|hai|ba|bốn|năm|lăm|sáu|bảy|tám|chín))?\b'
    normalized = re.sub(
        pattern_teens,
        lambda m: f"1{digit_map[m.group(1).lower()]}" if m.group(1) else "10",
        normalized,
        flags=re.IGNORECASE
    )

    # 6. Xử lý số đơn lẻ đi sau từ khóa chỉ địa chỉ (số, ngõ, hẻm, khu, quận)
    pattern_single_after_kw = r'\b(số|ngõ|ngách|hẻm|khu|quận|phường)\s+(một|hai|ba|bốn|năm|sáu|bảy|tám|chín)\b'
    normalized = re.sub(
        pattern_single_after_kw,
        lambda m: f"{m.group(1)} {digit_map[m.group(2).lower()]}",
        normalized,
        flags=re.IGNORECASE
    )
    return normalized


# Danh sách các cụm từ nhiễu phổ biến do Prompt Echo hoặc Hallucination của Qwen2-0.5B
HALLUCINATED_PHRASES = [
    "bắt buộc", "bát buộc", "quán bát buộc", "quán bắt buộc",
    "123 nguyễn du", "123 đường nguyễn văn cừ", "123 nguyễn văn cừ", "123 nguyễn trãi",
    "địa chỉ cụ thể không rõ", "không có thông tin", "chữ tên quán đọc được",
    "chữ địa chỉ", "trích nguyên văn", "những quán ăn ngon", "các quán ăn ngon",
    "quán ăn ngon", "đến hà nội thì ăn gì", "ăn gì ở", "review quán"
]

def clean_extracted_value(val: Optional[str], field_type: str = "general") -> Optional[str]:
    """Làm sạch và lọc bỏ các giá trị rác, prompt echo, chỉ số confidence DBNet hoặc hallucination"""
    if val is None:
        return None
    val_str = str(val).strip().strip('"\'.,;-:')
    if not val_str:
        return None
        
    val_lower = val_str.lower()
    
    # 1. Lọc các token rỗng/null cơ bản
    invalid_exact = {
        "null", "none", "chưa rõ", "không rõ", "không có", "n/a", "na",
        "unknown", "<null>", "không", "trống", "follow", "like", "share"
    }
    if val_lower in invalid_exact or val_str.startswith("<"):
        return None
        
    # 2. Lọc các cụm từ hallucination / prompt echo
    for phrase in HALLUCINATED_PHRASES:
        if phrase in val_lower:
            return None
            
    # 3. Xóa các chỉ số confidence/bounding box của DBNet bị lọt vào đầu/cuối chuỗi (VD: "0.81 THỊT XIÊN...", "10.75%")
    val_str = re.sub(r'^(?:\d+\.\d+%?\s*)+', '', val_str).strip()
    val_str = re.sub(r'\b(?:Area|DT)\s*:\s*\d+(?:\.\d+)?%?\b', '', val_str, flags=re.IGNORECASE).strip()
    if not val_str:
        return None
    val_lower = val_str.lower()
        
    # 4. Nếu chuỗi chỉ toàn số, phần trăm, giờ giấc (VD: "1186", "10.75%", "17:00 - 20:00") -> loại bỏ
    if re.match(r'^[\d\s.,:%+-]+$', val_str):
        return None
        
    # 5. Kiểm tra riêng cho trường tên quán: không lấy câu mô tả quá dài (> 8 từ) hoặc chứa từ khóa so sánh
    if field_type == "name":
        descriptive_Stopwords = ["đông khách nhất", "ngon nhất", "rẻ nhất", "nổi tiếng nhất", "bình dân đông khách"]
        if any(sw in val_lower for sw in descriptive_Stopwords):
            return None
        # Nếu tên quán bị nhận nhầm là một địa chỉ thuần túy (VD: "586 Trường Sa, Phú Nhuận")
        if re.match(r'^\d+[A-Za-z]?\s+(?:đường|phố|ngõ|hẻm)?\s*[A-ZÀ-Ỹ]', val_str) and ("," in val_str):
            return None
            
    # 6. Kiểm tra riêng cho trường địa chỉ: phải có độ dài tối thiểu và không phải tên món ăn đơn thuần
    if field_type == "address":
        if len(val_str) < 4:
            return None
            
    return val_str


def extract_regex_anchors(text: str) -> dict:
    """Dùng Regex bắt nhanh các cụm từ khóa chỉ tên quán/địa chỉ trong văn bản STT"""
    name_hint, addr_hint = None, None
    
    # Bắt tên quán sau các cụm giới thiệu trực tiếp
    m_name = re.search(
        r'(?:đây là quán|quán tên là|tại quán|ở quán|chỗ)\s+([^.,\n]{2,35}?)(?:\s+ở|\s+tại|\s+số|\s+ngõ|\s+đường|\s+phố|\s+quận|\s+mở|\s+nha|\s+nhé|$)',
        text, re.IGNORECASE
    )
    if m_name:
        cand = clean_extracted_value(m_name.group(1).strip(), field_type="name")
        if cand and len(cand.split()) <= 7:
            name_hint = cand
        
    # Bắt cụm địa chỉ (số ... ngõ/phố/đường...)
    m_addr = re.search(
        r'((?:số\s+\d+|ngõ\s+\d+|hẻm\s+\d+|\d+\s+[a-zà-ỹ]+|đường\s+[^.,\n]{3,30}|phố\s+[^.,\n]{3,30}|khu\s+\d+[^.,\n]{3,35})[^.,\n]{0,35}?)(?:\s+quán mở|\s+mở cửa|\s+nha|\s+nhé|\s+mọi người|\s+các bạn|$)',
        text, re.IGNORECASE
    )
    if m_addr:
        addr_hint = clean_extracted_value(m_addr.group(1).strip(), field_type="address")
        
    return {"regex_name": name_hint, "regex_addr": addr_hint}


def parse_json_output(text: str) -> dict:
    """Trích xuất dict từ chuỗi JSON và lọc bỏ các giá trị rỗng/nhiễu"""
    raw_dict = {}
    try:
        match = re.search(r"```(?:json)?\s*([\s\S]*?)\s*```", text)
        raw_json = match.group(1).strip() if match else None
        if not raw_json:
            match = re.search(r"(\{[\s\S]*\})", text)
            raw_json = match.group(1).strip() if match else text.strip()
        raw_dict = json.loads(raw_json)
    except Exception:
        name_match = re.search(r'["\'](?:tên_quán|tên_quán_tiềm_năng)["\']\s*:\s*["\']([^"\']+)["\']', text)
        addr_match = re.search(r'["\'](?:địa_chỉ|địa_chỉ_tiềm_năng)["\']\s*:\s*["\']([^"\']+)["\']', text)
        other_match = re.search(r'["\']chữ_khác_trên_ảnh["\']\s*:\s*["\']([^"\']+)["\']', text)
        raw_dict = {
            "tên_quán": name_match.group(1) if name_match else None,
            "địa_chỉ": addr_match.group(1) if addr_match else None,
            "chữ_khác_trên_ảnh": other_match.group(1) if other_match else None,
        }

    cleaned = {}
    for k, v in raw_dict.items():
        if isinstance(v, dict):
            cleaned[k] = v
        elif "tên_quán" in k:
            cleaned[k] = clean_extracted_value(v, field_type="name")
        elif "địa_chỉ" in k:
            cleaned[k] = clean_extracted_value(v, field_type="address")
        else:
            cleaned[k] = clean_extracted_value(v, field_type="general")
    return cleaned


# =============================================================================
# CLASS CỐT LÕI ĐÃ FIX LỖI KẾT LUẬN NULL / SAI: VinternVideoPOIAnalyzer
# =============================================================================

class VinternVideoPOIAnalyzer:
    def __init__(self, model_id: str = "5CD-AI/Vintern-1B-v3_5"):
        print(f"[*] Đang tải mô hình VLM: {model_id}...")
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.dtype = (
            torch.bfloat16
            if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
            else (torch.float16 if torch.cuda.is_available() else torch.float32)
        )
        
        self.model = AutoModel.from_pretrained(
            model_id,
            torch_dtype=self.dtype,
            low_cpu_mem_usage=True,
            trust_remote_code=True,
        ).eval()
        
        if torch.cuda.is_available():
            self.model = self.model.to(self.device)
            
        self.tokenizer = AutoTokenizer.from_pretrained(
            model_id,
            trust_remote_code=True,
            use_fast=False
        )
        self.transform = build_transform(input_size=448)
        
        self.gen_config = dict(
            max_new_tokens=200,
            do_sample=False,
            repetition_penalty=1.05
        )
        
        self.current_context = None
        self.frame_evidence = []
        print("[✓] Tải mô hình và Tokenizer thành công!\n")

    def _select_best_ocr_candidate(self, field: str) -> tuple:
        """
        Bầu chọn (Majority Voting + Quality Scoring) ứng viên tốt nhất từ danh sách frame_evidence.
        Trả về (best_value, frame_source_str)
        """
        candidates = []
        for e in self.frame_evidence:
            val = e.get(field)
            if val:
                candidates.append((val, e["frame_idx"]))
                
        if not candidates:
            return None, None

        # Nhóm các ứng viên tương đồng (không phân biệt hoa thường)
        norm_counts = Counter(c[0].lower().strip() for c in candidates)

        def score_candidate(item):
            val, f_idx = item
            v_low = val.lower().strip()
            freq = norm_counts[v_low]
            score = freq * 10.0
            
            if field == "địa_chỉ":
                # Ưu tiên địa chỉ có số nhà và tên đường/quận rõ ràng
                if re.search(r'\d+', val):
                    score += 8.0
                if any(kw in v_low for kw in ["phố", "đường", "ngõ", "hẻm", "quận", "phường", "khu", "tp", "hà nội", "phú nhuận", "ba đình", "đống đa", "long biên", "bắc ninh"]):
                    score += 6.0
                score += min(len(val), 45) * 0.15
            elif field == "tên_quán":
                # Ưu tiên tên quán có độ dài vừa phải (2 - 6 từ)
                word_cnt = len(val.split())
                if 2 <= word_cnt <= 6:
                    score += 6.0
                if any(kw in v_low for kw in ["quán", "tiệm", "nhà hàng", "ốc", "bún", "phở", "lẩu", "nướng", "cafe", "bánh"]):
                    score += 3.0
                score += min(len(val), 30) * 0.1
            return score

        best_val, best_frame = max(candidates, key=score_candidate)
        return best_val, f"Frame {best_frame} (OCR)"

    def init_video_context(
        self, 
        script: str = "", 
        comments: Union[str, List[str]] = None, 
        video_id: Optional[str] = None
    ) -> dict:
        """BƯỚC 1: Trích xuất nguyên văn từ Script & Comments, kết hợp chuẩn hóa số và Regex."""
        if isinstance(comments, list):
            comments_text = "\n".join([f"- {c}" for c in comments if c])
        elif isinstance(comments, str):
            comments_text = comments.strip()
        else:
            comments_text = ""

        raw_script = script.strip() if script else ""
        norm_script = normalize_stt_text(raw_script)
        anchors = extract_regex_anchors(norm_script)

        cand_name, cand_addr = None, None
        if norm_script or comments_text:
            prompt = f'''Trích xuất nguyên văn Tên quán ăn và Địa chỉ được nhắc đến trong văn bản dưới đây.
Quy tắc:
- Chỉ trích xuất thông tin thực sự có trong văn bản, không tự suy diễn.
- Không lấy câu mô tả dài (ví dụ: "quán đông khách nhất...") làm tên quán.
- Nếu không có tên quán hoặc địa chỉ cụ thể, điền null.

Lời thoại: "{norm_script if norm_script else 'Không có'}"
Bình luận: "{comments_text if comments_text else 'Không có'}"

Trả về JSON:
{{
  "tên_quán_tiềm_năng": "<tên quán hoặc null>",
  "địa_chỉ_tiềm_năng": "<số nhà, tên đường/phố, quận hoặc null>"
}}'''

            with torch.no_grad():
                response, _ = self.model.chat(
                    self.tokenizer,
                    pixel_values=None,
                    question=prompt,
                    generation_config=self.gen_config,
                    history=None,
                    return_history=True
                )
            parsed = parse_json_output(response)
            cand_name = parsed.get("tên_quán_tiềm_năng")
            cand_addr = parsed.get("địa_chỉ_tiềm_năng")
            
            # Kiểm tra chống Hallucination ngay ở Bước 1: kết quả phải tồn tại trong văn bản gốc
            combined_text_low = f"{norm_script} {comments_text}".lower()
            if cand_name:
                name_tokens = [w for w in re.findall(r'\w+', cand_name.lower()) if len(w) > 1]
                if name_tokens and sum(1 for w in name_tokens if w in combined_text_low) / len(name_tokens) < 0.5:
                    cand_name = None
            if cand_addr:
                addr_tokens = [w for w in re.findall(r'\w+', cand_addr.lower()) if len(w) > 1]
                if addr_tokens and sum(1 for w in addr_tokens if w in combined_text_low) / len(addr_tokens) < 0.5:
                    cand_addr = None

        # Bổ sung từ Regex nếu LLM bỏ sót
        cand_name = cand_name or anchors["regex_name"]
        cand_addr = cand_addr or anchors["regex_addr"]

        self.current_context = {
            "video_id": video_id,
            "script": norm_script,
            "comments": comments_text,
            "preliminary_clues": {
                "tên_quán_tiềm_năng": cand_name,
                "địa_chỉ_tiềm_năng": cand_addr
            }
        }
        self.frame_evidence = []
        
        print(f"[1. KHỞI TẠO NGỮ CẢNH] Video ID: {video_id or 'Unknown'}")
        print(f"  • Tên quán tiềm năng (Text) : {cand_name}")
        print(f"  • Địa chỉ tiềm năng (Text)  : {cand_addr}")
        return self.current_context

    def process_frame(
        self, 
        frame: Union[Image.Image, np.ndarray, str], 
        frame_idx: int = 1, 
        total_frames: int = 1
    ) -> dict:
        """
        BƯỚC 2: OCR TRÊN TỪNG KHUNG HÌNH.
        Sử dụng prompt trung tính (không dùng chữ IN HOA như 'BẮT BUỘC' để tránh mô hình 0.5B copy lại chữ trong prompt).
        """
        if self.current_context is None:
            raise ValueError("Chưa khởi tạo ngữ cảnh! Hãy gọi init_video_context(...) trước.")
            
        if isinstance(frame, str):
            frame_img = Image.open(frame).convert("RGB")
        elif isinstance(frame, np.ndarray):
            frame_img = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        else:
            frame_img = frame.convert("RGB")
            
        patches = dynamic_preprocess(frame_img, image_size=448, use_thumbnail=True, max_num=3)
        pixel_tensors = [self.transform(patch) for patch in patches]
        pixel_tensor = torch.stack(pixel_tensors).to(dtype=self.dtype, device=self.device)
        num_patches_list = [pixel_tensor.size(0)]
        
        prompt = '''<image>
Hãy đọc văn bản tiếng Việt xuất hiện trên biển hiệu hoặc phụ đề trong ảnh để tìm tên quán và địa chỉ.
Lưu ý:
- Chỉ ghi lại chữ thực sự nhìn thấy rõ trong ảnh.
- Bỏ qua các con số thập phân nhỏ (như 0.81, 0.92) hoặc phần trăm diện tích.
- Nếu không có tên quán hoặc địa chỉ trong ảnh, hãy để giá trị là null.

Trả về JSON:
{
  "tên_quán": null,
  "địa_chỉ": null,
  "chữ_khác_trên_ảnh": null
}'''

        with torch.no_grad():
            response, _ = self.model.chat(
                self.tokenizer,
                pixel_values=pixel_tensor,
                question=prompt,
                generation_config=self.gen_config,
                num_patches_list=num_patches_list,
                history=None,
                return_history=True
            )
            
        parsed = parse_json_output(response)
        
        # Cứu dữ liệu nếu model đặt địa chỉ vào ô tên_quán hoặc chữ_khác_trên_ảnh
        ocr_name = parsed.get("tên_quán")
        ocr_addr = parsed.get("địa_chỉ")
        ocr_other = parsed.get("chữ_khác_trên_ảnh")
        
        if not ocr_addr and ocr_other:
            if re.search(r'(?:đc|địa chỉ|số|ngõ|phố|đường|quận)\s*[:.]?\s*\d*', ocr_other, re.IGNORECASE):
                ocr_addr = clean_extracted_value(re.sub(r'^(?:đc|địa chỉ)\s*[:.]?\s*', '', ocr_other, flags=re.IGNORECASE), "address")

        evidence_entry = {
            "frame_idx": frame_idx,
            "tên_quán": ocr_name,
            "địa_chỉ": ocr_addr,
            "đặc_trưng": ocr_other,
            "raw_response": response
        }
        self.frame_evidence.append(evidence_entry)
        
        print(f"  [Frame {frame_idx}/{total_frames}] -> Tên OCR: {evidence_entry['tên_quán']} | Địa chỉ OCR: {evidence_entry['địa_chỉ']} | Chữ khác: {evidence_entry['đặc_trưng']}")
        return evidence_entry

    def conclude(self) -> dict:
        """
        BƯỚC 3: TỔNG HỢP & KẾT LUẬN CUỐI CÙNG.
        - Đưa kết quả OCR từ các frame lên làm NGUỒN 1 (Ưu tiên cao nhất).
        - Áp dụng cơ chế Majority Voting + Quality Scoring từ OCR làm chuẩn đối chiếu để không bao giờ bị mất dữ liệu (None) khi các frame đã đọc được chữ.
        """
        if self.current_context is None:
            raise ValueError("Không có ngữ cảnh để tổng hợp kết luận.")
            
        text_name = self.current_context["preliminary_clues"].get("tên_quán_tiềm_năng")
        text_addr = self.current_context["preliminary_clues"].get("địa_chỉ_tiềm_năng")
        
        # 1. Tìm ứng viên OCR tốt nhất từ các frame bằng thuật toán chấm điểm & bầu chọn
        best_ocr_name, ocr_name_src = self._select_best_ocr_candidate("tên_quán")
        best_ocr_addr, ocr_addr_src = self._select_best_ocr_candidate("địa_chỉ")
        
        # 2. Tổng hợp danh sách các frame hợp lệ cho prompt
        valid_frames_info = []
        for e in self.frame_evidence:
            parts = []
            if e.get("tên_quán"):
                parts.append(f"Tên quán='{e['tên_quán']}'")
            if e.get("địa_chỉ"):
                parts.append(f"Địa chỉ='{e['địa_chỉ']}'")
            if e.get("đặc_trưng"):
                parts.append(f"Chữ khác='{e['đặc_trưng']}'")
            if parts:
                valid_frames_info.append(f"- Frame {e['frame_idx']}: " + ", ".join(parts))
        
        frames_text = "\n".join(valid_frames_info) if valid_frames_info else "Không có chữ trên ảnh."

        # Xây dựng khối thông tin văn bản (chỉ thêm vào nếu khác None để tránh neo chữ 'null')
        text_clues_lines = []
        if text_name:
            text_clues_lines.append(f"- Tên quán từ lời thoại: {text_name}")
        if text_addr:
            text_clues_lines.append(f"- Địa chỉ từ lời thoại: {text_addr}")
        if self.current_context.get("script"):
            text_clues_lines.append(f"- Lời thoại gốc: \"{self.current_context['script']}\"")
        text_section = "\n".join(text_clues_lines) if text_clues_lines else "Không có thông tin từ lời thoại."

        prompt = f'''Dựa vào dữ liệu đã thu thập dưới đây, hãy xác định TÊN QUÁN và ĐỊA CHỈ chính xác nhất.

NGUỒN 1 - Chữ đọc trực tiếp từ biển hiệu trên ảnh (ĐỘ CHÍNH XÁC CAO NHẤT):
{frames_text}
- Tên quán rõ nhất trên ảnh: {best_ocr_name or 'Chưa có'}
- Địa chỉ rõ nhất trên ảnh: {best_ocr_addr or 'Chưa có'}

NGUỒN 2 - Thông tin bổ trợ từ lời thoại:
{text_section}

Quy tắc:
1. Ưu tiên tuyệt đối chữ đọc từ biển hiệu ở NGUỒN 1 karena viết đúng chính tả và số nhà.
2. Chỉ dùng NGUỒN 2 khi NGUỒN 1 bị khuyết thiếu.
3. Không tự bịa thêm tên đường hay thành phố không có trong 2 nguồn.

Trả về JSON:
{{
  "tên_quán": "<Tên quán chính xác nhất>",
  "địa_chỉ": "<Địa chỉ chính xác nhất>",
  "độ_tin_cậy": "Cao"
}}'''

        with torch.no_grad():
            response, _ = self.model.chat(
                self.tokenizer,
                pixel_values=None,
                question=prompt,
                generation_config=self.gen_config,
                history=None,
                return_history=True
            )
            
        final_result = parse_json_output(response)
        
        # 3. KIỂM TRA CHÉO & QUYẾT ĐỊNH CUỐI CÙNG (DETERMINISTIC GROUNDING & OCR PRIORITY)
        combined_source = (
            f"{self.current_context.get('script', '')} "
            f"{self.current_context.get('comments', '')} "
            f"{frames_text}"
        ).lower()

        pred_name = final_result.get("tên_quán")
        pred_addr = final_result.get("địa_chỉ")

        # Kiểm tra xem kết quả của LLM có bịa ra từ không nằm trong nguồn không
        if pred_name:
            words = [w for w in re.findall(r'\w+', pred_name.lower()) if len(w) > 1]
            if words and (sum(1 for w in words if w in combined_source) / len(words)) < 0.5:
                pred_name = None
        if pred_addr:
            words = [w for w in re.findall(r'\w+', pred_addr.lower()) if len(w) > 1]
            if words and (sum(1 for w in words if w in combined_source) / len(words)) < 0.5:
                pred_addr = None

        # Nếu đã có best_ocr_name / best_ocr_addr rõ ràng từ bảng hiệu, ưu tiên chuẩn hóa theo OCR
        # (Khắc phục lỗi LLM copy câu mô tả dài hoặc số viết bằng chữ từ Nguồn 2)
        name_source = "Chưa rõ"
        addr_source = "Chưa rõ"

        if best_ocr_name:
            # Nếu pred_name rỗng, hoặc pred_name đang lấy câu văn bản kém chính xác hơn OCR
            if not pred_name or (text_name and pred_name.lower() == text_name.lower() and best_ocr_name.lower() != text_name.lower()):
                pred_name = best_ocr_name
                name_source = ocr_name_src
            else:
                name_source = ocr_name_src if pred_name.lower() in best_ocr_name.lower() or best_ocr_name.lower() in pred_name.lower() else "Lời thoại + OCR"
        elif pred_name:
            name_source = "Lời thoại (Script)"
        elif text_name:
            pred_name = text_name
            name_source = "Lời thoại (Script)"

        if best_ocr_addr:
            # Nếu pred_addr rỗng, hoặc pred_addr không có chữ số trong khi best_ocr_addr có số nhà rõ ràng
            if not pred_addr or (re.search(r'\d+', best_ocr_addr) and not re.search(r'\d+', pred_addr)) or (text_addr and pred_addr.lower() == text_addr.lower() and best_ocr_addr.lower() != text_addr.lower()):
                pred_addr = best_ocr_addr
                addr_source = ocr_addr_src
            else:
                addr_source = ocr_addr_src if pred_addr.lower() in best_ocr_addr.lower() or best_ocr_addr.lower() in pred_addr.lower() else "Lời thoại + OCR"
        elif pred_addr:
            addr_source = "Lời thoại (Script)"
        elif text_addr:
            pred_addr = text_addr
            addr_source = "Lời thoại (Script)"

        # Tính toán độ tin cậy thực tế dựa trên bằng chứng thu được
        if pred_name and pred_addr:
            confidence = "Cao" if (best_ocr_name or best_ocr_addr) else "Trung bình"
        elif pred_name or pred_addr:
            confidence = "Trung bình"
        else:
            confidence = "Thấp"

        final_result["tên_quán"] = pred_name
        final_result["địa_chỉ"] = pred_addr
        final_result["độ_tin_cậy"] = confidence
        final_result["dẫn_chứng"] = {
            "nguồn_tên_quán": name_source if pred_name else None,
            "nguồn_địa_chỉ": addr_source if pred_addr else None
        }

        print("\n" + "="*50)
        print("🎯 KẾT LUẬN CUỐI CÙNG:")
        print(f"  • Tên quán   : {final_result.get('tên_quán')}")
        print(f"  • Địa chỉ    : {final_result.get('địa_chỉ')}")
        print(f"  • Độ tin cậy : {final_result.get('độ_tin_cậy')}")
        print(f"  • Dẫn chứng  : {json.dumps(final_result.get('dẫn_chứng', {}), ensure_ascii=False)}")
        print("="*50)
        
        final_result["chi_tiết_frames"] = [
            {
                "frame_idx": e.get("frame_idx"),
                "tên_quán_ocr": e.get("tên_quán"),
                "địa_chỉ_ocr": e.get("địa_chỉ"),
                "chữ_khác_ocr": e.get("đặc_trưng")
            }
            for e in self.frame_evidence
        ]
        
        return final_result

    def clear_context(self):
        """BƯỚC 4: Xóa sạch ngữ cảnh và giải phóng VRAM."""
        self.current_context = None
        self.frame_evidence = []
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        print("[✓] Đã xóa sạch ngữ cảnh & dọn dẹp bộ nhớ GPU.\n")

    def analyze_video(
        self,
        frames: List[Union[Image.Image, np.ndarray, str]],
        script: str = "",
        comments: Union[str, List[str]] = None,
        video_id: Optional[str] = None,
        verbose: bool = True
    ) -> dict:
        """Quy trình xử lý hoàn chỉnh 4 bước cho 1 video."""
        self.init_video_context(script=script, comments=comments, video_id=video_id)
        total = len(frames)
        if verbose:
            print(f"\n[XỬ LÝ {total} KHUNG HÌNH CHO VIDEO: {video_id or 'Unknown'}]")
        for i, frame in enumerate(frames):
            self.process_frame(frame, frame_idx=i + 1, total_frames=total)
        result = self.conclude()
        self.clear_context()
        return result

In [5]:
# Khởi tạo mô hình Vintern-1B
analyzer = VinternVideoPOIAnalyzer("5CD-AI/Vintern-1B-v3_5")

[*] Đang tải mô hình VLM: 5CD-AI/Vintern-1B-v3_5...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

configuration_internvl_chat.py: 0.00B [00:00, ?B/s]

configuration_intern_vit.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/5CD-AI/Vintern-1B-v3_5:
- configuration_intern_vit.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/5CD-AI/Vintern-1B-v3_5:
- configuration_internvl_chat.py
- configuration_intern_vit.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
`torch_dtype` is deprecated! Use `dtype` instead!


... (10 outputs left out)

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/790 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/744 [00:00<?, ?B/s]

[✓] Tải mô hình và Tokenizer thành công!



## 🧪 Demo Thực Thi Pipeline
Dưới đây là 2 cách thực thi:
- **Cách 1: Thực thi chi tiết từng bước** cho 1 video để quan sát rõ luồng dữ liệu.
- **Cách 2: Thực thi tự động hàng loạt video (Batch Processing)** đọc từ CSV và thư mục frame (`skip_existing=False` mặc định để chạy lại các kết quả từng bị lỗi ở bản cũ).

In [6]:
# =============================================================================
# CẤU HÌNH ĐƯỜNG DẪN MÔI TRƯỜNG (GOOGLE COLAB & LOCAL WINDOWS)
# =============================================================================
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    WORKSPACE_DIR = "/content/drive/MyDrive/colab/pbl/main"
    VIDEOS_DIR = "/content/drive/MyDrive/colab/PBL_Build_Data/videos"
except Exception:
    WORKSPACE_DIR = os.path.dirname(os.path.abspath("__file__")) if "__file__" in locals() else r"G:\マイドライブ\colab\pbl\main"
    VIDEOS_DIR = r"G:\マイドライブ\colab\PBL_Build_Data\videos"

if not os.path.exists(VIDEOS_DIR):
    alt_vids = r"G:\マイドライブ\colab\PBL_Build_Data\videos"
    if os.path.exists(alt_vids):
        VIDEOS_DIR = alt_vids

CSV_PATH = os.path.join(WORKSPACE_DIR, "video_speech_to_text.csv")
OUTPUT_DIR = os.path.join(WORKSPACE_DIR, "output_results")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("=" * 65)
print(f"Thư mục làm việc (WORKSPACE_DIR) : {WORKSPACE_DIR}")
print(f"Thư mục chứa video (VIDEOS_DIR)  : {VIDEOS_DIR}")
print(f"File kịch bản CSV (CSV_PATH)     : {CSV_PATH}")
print(f"Thư mục lưu kết quả (OUTPUT_DIR) : {OUTPUT_DIR}")
print(f"Tồn tại thư mục video            : {os.path.exists(VIDEOS_DIR)}")
print(f"Tồn tại file CSV                 : {os.path.exists(CSV_PATH)}")
print("=" * 65)

Mounted at /content/drive
Thư mục làm việc (WORKSPACE_DIR) : /content/drive/MyDrive/colab/pbl/main
Thư mục chứa video (VIDEOS_DIR)  : /content/drive/MyDrive/colab/PBL_Build_Data/videos
File kịch bản CSV (CSV_PATH)     : /content/drive/MyDrive/colab/pbl/main/video_speech_to_text.csv
Thư mục lưu kết quả (OUTPUT_DIR) : /content/drive/MyDrive/colab/pbl/main/output_results
Tồn tại thư mục video            : True
Tồn tại file CSV                 : True


In [7]:
# =============================================================================
# CÁCH 1: DEMO TỪNG BƯỚC CHO 1 VIDEO CỤ THỂ
# =============================================================================

sample_video_id = "7154239414439349510"
sample_video_path = os.path.join(VIDEOS_DIR, f"{sample_video_id}.mp4")

sample_script = ""
if os.path.exists(CSV_PATH):
    import csv
    with open(CSV_PATH, "r", encoding="utf-8-sig") as f:
        reader = csv.DictReader(f)
        for row in reader:
            if row.get("video_id") == sample_video_id:
                sample_script = row.get("text", "")
                break

if not sample_script:
    sample_script = (
        "đến hà nội thì ăn gì phần tiếp theo với mọi người... ăn xong thì mình chạy qua chỗ ốc nóng hài trang ở số một a đinh liền..."
    )

sample_comments = []

print(f"[*] Đang tải khung hình cho video {sample_video_id}...")
selected_frames = get_video_keyframes(
    video_path=sample_video_path,
    max_keyframes=5,
    video_id=sample_video_id,
    workspace_dir=WORKSPACE_DIR
)

if not selected_frames:
    print("[*] Không tìm thấy video hay frame cục bộ, tạo 2 khung hình mẫu để test luồng:")
    dummy_img = Image.new('RGB', (640, 480), color=(240, 240, 240))
    selected_frames = [dummy_img, dummy_img]

sample_out_dir = os.path.join(OUTPUT_DIR, sample_video_id)
sample_frames_dir = os.path.join(sample_out_dir, "frames")
os.makedirs(sample_frames_dir, exist_ok=True)
saved_frame_files = []
for i, f in enumerate(selected_frames, 1):
    f_name = f"frame_{i:02d}.jpg"
    f_path = os.path.join(sample_frames_dir, f_name)
    try:
        if isinstance(f, Image.Image):
            f.convert("RGB").save(f_path, "JPEG", quality=95)
        elif isinstance(f, np.ndarray):
            ret, buf = cv2.imencode(".jpg", f)
            if ret:
                buf.tofile(f_path)
        saved_frame_files.append(os.path.join("frames", f_name))
    except Exception as fe:
        print(f"Lỗi lưu frame {f_name}: {fe}")
print(f"[*] Đã lưu {len(saved_frame_files)} frame vào: {sample_frames_dir}")

# Bước 1: Khởi tạo ngữ cảnh
analyzer.init_video_context(script=sample_script, comments=sample_comments, video_id=sample_video_id)

# Bước 2: Xử lý tuần tự từng frame
total = len(selected_frames)
print(f"\n[2. XỬ LÝ TUẦN TỰ {total} KHUNG HÌNH]")
for i, frame in enumerate(selected_frames):
    analyzer.process_frame(frame, frame_idx=i + 1, total_frames=total)

# Bước 3: Kết luận
result = analyzer.conclude()
result["video_id"] = sample_video_id
result["frames_used"] = len(selected_frames)
result["frames_dir"] = os.path.join(sample_video_id, "frames")
result["saved_frames"] = saved_frame_files

with open(os.path.join(sample_out_dir, "result.json"), "w", encoding="utf-8") as wf:
    json.dump(result, wf, ensure_ascii=False, indent=2)

# Bước 4: Xóa ngữ cảnh
analyzer.clear_context()

[*] Đang tải khung hình cho video 7154239414439349510...


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


[*] Đã lưu 5 frame vào: /content/drive/MyDrive/colab/pbl/main/output_results/7154239414439349510/frames
[1. KHỞI TẠO NGỮ CẢNH] Video ID: 7154239414439349510
  • Tên quán tiềm năng (Text) : ốc nóng hài trang
  • Địa chỉ tiềm năng (Text)  : None

[2. XỬ LÝ TUẦN TỰ 5 KHUNG HÌNH]


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


  [Frame 1/5] -> Tên OCR: Hà Trang | Địa chỉ OCR: 1A Đinh Liệt | Chữ khác: None


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


... (3 outputs left out)

Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


  [Frame 4/5] -> Tên OCR: Ốc Nóng Hồ Trang | Địa chỉ OCR: 1A Đinh Liệt | Chữ khác: None


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


  [Frame 5/5] -> Tên OCR: Ốc Nóng Hà Trang | Địa chỉ OCR: 1A Đình Liệt | Chữ khác: None

🎯 KẾT LUẬN CUỐI CÙNG:
  • Tên quán   : Ốc Nóng Hà Trang
  • Địa chỉ    : 1A Đinh Liệt
  • Độ tin cậy : Cao
  • Dẫn chứng  : {"nguồn_tên_quán": "Frame 3 (OCR)", "nguồn_địa_chỉ": "Frame 1 (OCR)"}
[✓] Đã xóa sạch ngữ cảnh & dọn dẹp bộ nhớ GPU.



In [8]:
# =============================================================================
# CÁCH 2: XỬ LÝ HÀNG LOẠT THEO BATCH (BATCH POI PIPELINE)
# =============================================================================
import os
import csv
import time
import json
import pandas as pd
from typing import List, Dict, Any, Optional

def run_batch_poi_pipeline(
    csv_path: Optional[str] = None,
    videos_dir: Optional[str] = None,
    output_dir: Optional[str] = None,
    max_videos: Optional[int] = 10,
    start_index: int = 0,
    video_id_filter: Optional[str] = None,
    max_keyframes_per_video: int = 5,
    sample_fps: float = 1.0,
    skip_existing: bool = False
) -> List[Dict[str, Any]]:
    target_csv = csv_path or CSV_PATH
    target_vids_dir = videos_dir or VIDEOS_DIR
    target_out_dir = output_dir or OUTPUT_DIR
    
    os.makedirs(target_out_dir, exist_ok=True)
    
    if not os.path.exists(target_csv):
        print(f"[!] Không tìm thấy file CSV: {target_csv}")
        return []
        
    if not os.path.exists(target_vids_dir):
        print(f"[!] Không tìm thấy thư mục video: {target_vids_dir}")
        return []

    records = []
    with open(target_csv, "r", encoding="utf-8-sig") as f:
        reader = csv.DictReader(f)
        for row in reader:
            vid = row.get("video_id", "").strip()
            script_text = row.get("text", "").strip()
            status = row.get("status", "").strip()
            
            if vid and (status == "SUCCESS" or not status):
                records.append({
                    "video_id": vid,
                    "script": script_text,
                    "status_csv": status
                })

    print(f"[*] Đọc thành công {len(records)} dòng từ {os.path.basename(target_csv)}")

    if video_id_filter:
        records = [r for r in records if r["video_id"] == str(video_id_filter)]
        print(f"[*] Đã lọc theo video_id '{video_id_filter}': còn {len(records)} video")

    if start_index > 0:
        records = records[start_index:]
    if max_videos is not None and max_videos > 0:
        records = records[:max_videos]

    total_to_run = len(records)
    print(f"[*] Bắt đầu xử lý {total_to_run} video (Bắt đầu từ index {start_index})...\n")

    batch_start_time = time.time()
    results = []
    success_count = 0
    skipped_count = 0
    failed_count = 0

    for idx, item in enumerate(records, 1):
        vid = item["video_id"]
        script = item["script"]
        video_path = os.path.join(target_vids_dir, f"{vid}.mp4")
        video_out_dir = os.path.join(target_out_dir, vid)
        frames_out_dir = os.path.join(video_out_dir, "frames")
        result_json_path = os.path.join(video_out_dir, "result.json")

        print("=" * 65)
        print(f"[{idx}/{total_to_run}] VIDEO ID: {vid}")
        print("=" * 65)

        has_existing_frames = os.path.exists(frames_out_dir) and len(os.listdir(frames_out_dir)) > 0
        if skip_existing and os.path.exists(result_json_path) and has_existing_frames:
            try:
                with open(result_json_path, "r", encoding="utf-8") as rf:
                    cached_data = json.load(rf)
                print(f"  [SKIP] Video {vid} đã có kết quả và frames trước đó. Nạp từ cache:")
                print(f"  • Tên quán : {cached_data.get('tên_quán')}")
                print(f"  • Địa chỉ  : {cached_data.get('địa_chỉ')}")
                print(f"  • Thư mục frames: {cached_data.get('frames_dir', f'{vid}/frames')}\n")
                results.append(cached_data)
                skipped_count += 1
                continue
            except Exception:
                pass

        if not os.path.exists(video_path):
            print(f"  [!] Không tìm thấy file video: {video_path}")
            failed_count += 1
            results.append({
                "video_id": vid,
                "tên_quán": None,
                "địa_chỉ": None,
                "độ_tin_cậy": "Thấp",
                "status": "VIDEO_NOT_FOUND",
                "error": f"File not found: {video_path}"
            })
            continue

        try:
            t0 = time.time()
            
            frames = get_video_keyframes(
                video_path=video_path,
                max_keyframes=max_keyframes_per_video,
                video_id=vid,
                workspace_dir=WORKSPACE_DIR
            )

            if not frames:
                print(f"  [!] Không trích xuất được khung hình nào từ video {vid}")
                frames = [Image.new('RGB', (448, 448), color=(220, 220, 220))]

            os.makedirs(frames_out_dir, exist_ok=True)
            saved_frames_rel = []
            for f_idx, f_img in enumerate(frames, 1):
                frame_filename = f"frame_{f_idx:02d}.jpg"
                frame_save_path = os.path.join(frames_out_dir, frame_filename)
                try:
                    if isinstance(f_img, Image.Image):
                        f_img.convert("RGB").save(frame_save_path, "JPEG", quality=95)
                    elif isinstance(f_img, np.ndarray):
                        ret, buf = cv2.imencode(".jpg", f_img)
                        if ret:
                            buf.tofile(frame_save_path)
                    saved_frames_rel.append(os.path.join("frames", frame_filename))
                except Exception as fe:
                    print(f"  [!] Lỗi khi lưu frame {frame_filename}: {fe}")

            print(f"  [✓] Đã lưu {len(saved_frames_rel)} frames vào: {frames_out_dir}")

            res = analyzer.analyze_video(
                frames=frames,
                script=script,
                comments=[],
                video_id=vid,
                verbose=False
            )
            
            elapsed = time.time() - t0

            frames_detail = []
            ocr_evidence = res.get("chi_tiết_frames", [])
            for f_idx in range(len(frames)):
                f_name = f"frame_{f_idx+1:02d}.jpg"
                f_ev = ocr_evidence[f_idx] if f_idx < len(ocr_evidence) else {}
                frames_detail.append({
                    "frame_idx": f_idx + 1,
                    "image_file": os.path.join("frames", f_name),
                    "tên_quán_ocr": f_ev.get("tên_quán_ocr"),
                    "địa_chỉ_ocr": f_ev.get("địa_chỉ_ocr"),
                    "chữ_khác_ocr": f_ev.get("chữ_khác_ocr")
                })

            entry = {
                "video_id": vid,
                "tên_quán": res.get("tên_quán"),
                "địa_chỉ": res.get("địa_chỉ"),
                "độ_tin_cậy": res.get("độ_tin_cậy", "Chưa rõ"),
                "dẫn_chứng": res.get("dẫn_chứng", {}),
                "frames_used": len(frames),
                "frames_dir": os.path.join(vid, "frames"),
                "saved_frames": saved_frames_rel,
                "chi_tiết_frames": frames_detail,
                "time_seconds": round(elapsed, 2),
                "status": "SUCCESS"
            }
            results.append(entry)
            success_count += 1

            with open(result_json_path, "w", encoding="utf-8") as wf:
                json.dump(entry, wf, ensure_ascii=False, indent=2)

        except Exception as e:
            print(f"  [X] Lỗi khi xử lý video {vid}: {e}")
            failed_count += 1
            results.append({
                "video_id": vid,
                "tên_quán": None,
                "địa_chỉ": None,
                "status": "FAILED",
                "error": str(e)
            })

    total_batch_time = time.time() - batch_start_time

    summary_path = os.path.join(target_out_dir, "batch_vintern_results.json")
    with open(summary_path, "w", encoding="utf-8") as sf:
        json.dump(results, sf, ensure_ascii=False, indent=2)

    csv_out_path = os.path.join(target_out_dir, "batch_vintern_results.csv")
    csv_fields = ["video_id", "tên_quán", "địa_chỉ", "độ_tin_cậy", "frames_used", "frames_dir", "time_seconds", "status"]
    with open(csv_out_path, "w", encoding="utf-8-sig", newline="") as cf:
        writer = csv.DictWriter(cf, fieldnames=csv_fields, extrasaction="ignore")
        writer.writeheader()
        for r in results:
            writer.writerow(r)

    print("\n" + "#" * 65)
    print("HOÀN THÀNH QUY TRÌNH TEST BATCH!")
    print(f"• Tổng số video xử lý : {total_to_run}")
    print(f"• Thành công           : {success_count}")
    print(f"• Bỏ qua (đã có kết quả): {skipped_count}")
    print(f"• Thất bại             : {failed_count}")
    print(f"• Tổng thời gian       : {total_batch_time:.2f}s")
    print(f"• Thư mục lưu frames   : {target_out_dir}/<video_id>/frames/")
    print(f"• File CSV kết quả     : {csv_out_path}")
    print(f"• File JSON kết quả    : {summary_path}")
    print("#" * 65 + "\n")

    try:
        df = pd.DataFrame(results)
        cols_to_show = [c for c in ["video_id", "tên_quán", "địa_chỉ", "độ_tin_cậy", "frames_used", "frames_dir", "time_seconds", "status"] if c in df.columns]
        display(df[cols_to_show])
    except Exception:
        pass

    return results

# =============================================================================
# THỰC THI CHẠY TEST BATCH (skip_existing=False để ghi đè lại các kết quả cũ bị lỗi)
# =============================================================================
batch_results = run_batch_poi_pipeline(
    max_videos=10,
    start_index=0,
    max_keyframes_per_video=5,
    skip_existing=False
)

[*] Đọc thành công 1062 dòng từ video_speech_to_text.csv
[*] Bắt đầu xử lý 10 video (Bắt đầu từ index 0)...

[1/10] VIDEO ID: 6867816539274431746


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


  [✓] Đã lưu 5 frames vào: /content/drive/MyDrive/colab/pbl/main/output_results/6867816539274431746/frames
[1. KHỞI TẠO NGỮ CẢNH] Video ID: 6867816539274431746
  • Tên quán tiềm năng (Text) : None
  • Địa chỉ tiềm năng (Text)  : 40 ca


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


  [Frame 1/5] -> Tên OCR: None | Địa chỉ OCR: None | Chữ khác: LOAN: 0766233758


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


... (106 outputs left out)

  [Frame 4/5] -> Tên OCR: None | Địa chỉ OCR: None | Chữ khác: None


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


  [Frame 5/5] -> Tên OCR: Sen Hà Thành | Địa chỉ OCR: None | Chữ khác: None

🎯 KẾT LUẬN CUỐI CÙNG:
  • Tên quán   : Sen Hà Thành
  • Địa chỉ    : Cảng tàu khách Quốc tế Tuần Châu
  • Độ tin cậy : Cao
  • Dẫn chứng  : {"nguồn_tên_quán": "Frame 5 (OCR)", "nguồn_địa_chỉ": "Frame 2 (OCR)"}
[✓] Đã xóa sạch ngữ cảnh & dọn dẹp bộ nhớ GPU.


#################################################################
HOÀN THÀNH QUY TRÌNH TEST BATCH!
• Tổng số video xử lý : 10
• Thành công           : 10
• Bỏ qua (đã có kết quả): 0
• Thất bại             : 0
• Tổng thời gian       : 310.12s
• Thư mục lưu frames   : /content/drive/MyDrive/colab/pbl/main/output_results/<video_id>/frames/
• File CSV kết quả     : /content/drive/MyDrive/colab/pbl/main/output_results/batch_vintern_results.csv
• File JSON kết quả    : /content/drive/MyDrive/colab/pbl/main/output_results/batch_vintern_results.json
#################################################################



,video_id,tên_quán,địa_chỉ,độ_tin_cậy,frames_used,frames_dir,time_seconds,status
0,6867816539274431746,Bún đa trộn,40 ca,Trung bình,5,6867816539274431746/frames,31.06,SUCCESS
1,7109032785003695387,Cô Thủy,30 cửa nam,Cao,5,7109032785003695387/frames,29.93,SUCCESS
2,7126014523596672282,Cafe Ho,34 Ngọc Khánh - Ba Đình - Hà Nội,Cao,4,7126014523596672282/frames,25.58,SUCCESS
3,7129451774242606337,Bánh Bột Lọc Chan,"Chợ Gia Lâm, Ngõ 336 Ngọc Lâm, Long Biên",Cao,5,7129451774242606337/frames,34.58,SUCCESS
4,7136809431160917274,Thịt Xiên Nướng,"586 Trường Sa, Phú Nhuận",Cao,5,7136809431160917274/frames,32.35,SUCCESS
5,7142093601609223451,T.Cô Nhung,"Số 5 Âu Cơ, Khu 1, Tiền An, Bắc Ninh",Cao,3,7142093601609223451/frames,22.71,SUCCESS
6,7154239414439349510,Ốc Nóng Hà Trang,1A Đinh Liệt,Cao,5,7154239414439349510/frames,19.58,SUCCESS
7,7168667105418562818,Lẩu Nướng 1987,Khu 5 thị trấn thắng huyện Điệp Hoa,Trung bình,5,7168667105418562818/frames,33.06,SUCCESS
8,7173604513511673115,Quán Tùng,"178 Nguyễn Văn Tuyết, Đống Đa, Hà Nội",Cao,5,7173604513511673115/frames,34.31,SUCCESS
9,7180339417612078363,Sen Hà Thành,Cảng tàu khách Quốc tế Tuần Châu,Cao,5,7180339417612078363/frames,32.40,SUCCESS


## 📚 Phụ Lục: Cấu Hình PEFT LoRA Fine-Tuning (Tùy chọn)

In [9]:
def train_lora_setup(model):
    from peft import LoraConfig, get_peft_model
    
    if hasattr(model, 'vision_model'):
        for param in model.vision_model.parameters():
            param.requires_grad = False
            
    if hasattr(model, 'mlp1'):
        for param in model.mlp1.parameters():
            param.requires_grad = False
            
    config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM"
    )
    
    model.language_model = get_peft_model(model.language_model, config)
    model.language_model.print_trainable_parameters()
    return model